# Notebook 9 — BERT Text Classification: Fine-tuning a Transformer

**Module 3 · Text Classification**
*Author: Axel Sirota — Data Trainers LLC*

## Story (STAR)

- **Situation.** Your MLP from Notebook 8 hits ~92% accuracy on CNN headlines. That's solid. But the boss walks over: *"I heard GPT-4 can classify anything. Why don't we use one of those transformer things?"* You know the real reason your MLP plateaus — static Word2Vec embeddings treat "stocks soar" and "stocks soared" and "stocks are soaring" as three unrelated tokens. A transformer shares representations across inflections, tenses, and synonyms through **contextual** embeddings.
- **Task.** Fine-tune a pretrained BERT-family model (DistilBERT for speed) on the same 4-class headline dataset and compare apples-to-apples with the MLP. Ship the better one.
- **Action.** Install HuggingFace `transformers`, `datasets`, `accelerate`, and `evaluate`. Load `distilbert-base-uncased`. Tokenize with WordPiece subwords. Wrap the data in HF `Dataset`. Fine-tune with the `Trainer` API. Also implement a manual PyTorch loop (for when you need custom logic). Evaluate on the same held-out test set. Predict on new headlines.
- **Result.** You discover that transformer fine-tuning is NOT scary — it's ~40 lines of config. You learn when to reach for BERT vs. MLP, and you have a pattern you can apply to any classification problem in your day job.

## Learning objectives

By the end of this notebook you will be able to:

1. Articulate at a high level what attention does and why it helps (no need to derive the softmax math).
2. Describe BERT's architecture in 3 sentences: encoder-only, bidirectional context, pretrained on MLM + NSP.
3. Navigate the HuggingFace ecosystem: `transformers` (models + tokenizers), `datasets` (data wrangling), `evaluate` (metrics), `accelerate` (speedups).
4. Tokenize text with `AutoTokenizer` and understand what `input_ids`, `attention_mask`, `[CLS]`, `[SEP]`, `[PAD]` mean.
5. Load a pretrained model for classification with `AutoModelForSequenceClassification.from_pretrained(name, num_labels=K)`.
6. Fine-tune with the `Trainer` API (5 lines of config) and understand what it does under the hood.
7. Fine-tune with a manual PyTorch loop (for when `Trainer` isn't flexible enough).
8. Evaluate and compare with the MLP from Notebook 8 apples-to-apples.
9. Infer on new sentences (including adversarial examples).
10. Save, push to HuggingFace Hub (optional), and reload.

## Prerequisites

- Notebook 8 (MLP on CNN headlines) — the baseline we're beating.
- PyTorch fluency from Notebooks 5, 6, 8.
- Comfortable with training loops, DataLoaders, loss functions.

## Runtime

Recommended: **Colab with GPU** (T4 is plenty). Fine-tuning 2 epochs takes ~15 min on T4, ~3 hours on CPU. We strongly recommend GPU for this notebook.

## Section 0 — Environment Setup

Install dependencies. This cell pulls `transformers`, `datasets`, `accelerate`, `evaluate`, and `torch`. On a fresh Colab session, budget ~3–5 minutes the first time. Subsequent runs in the same session are instant.

> **Heads-up:** The first call to `AutoModel.from_pretrained('distilbert-base-uncased')` later downloads ~270 MB of model weights from HuggingFace and caches them under `~/.cache/huggingface/`. Budget a few extra seconds for that.

In [ ]:
# Install required packages for transformer fine-tuning.
# - transformers >= 4.40 brings AutoTokenizer, AutoModel, Trainer.
# - datasets >= 2.19 for HF Dataset abstraction and .map().
# - accelerate >= 0.28 is needed by Trainer for multi-GPU / fp16.
# - evaluate >= 0.4 for metrics (accuracy, F1).
# - torch >= 2.1 is our deep learning backbone.
# - scikit-learn for confusion matrix and stratified splits.
!pip install -q "transformers>=4.40" "datasets>=2.19" "accelerate>=0.28" \
    "evaluate>=0.4" "torch>=2.1" "scikit-learn>=1.3" "pandas>=2.0" \
    "matplotlib>=3.7" "seaborn>=0.13"

In [ ]:
# Imports — visualization, data, model, training (in that order).
import os
import random
import warnings
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from datasets import Dataset
import evaluate

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

warnings.filterwarnings("ignore")

# Reproducibility: seed every RNG we touch.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device detection: transformers will use CUDA automatically if available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version : {torch.__version__}")
print(f"Using device    : {device}")
if device.type == "cuda":
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

print("\nEnvironment ready.")

In [ ]:
# Hyperparameters for the whole notebook — defined up front, reused everywhere.
SEED          = 42
MODEL_NAME    = "distilbert-base-uncased"  # 6 layers, 66M params — Colab-friendly
# Alternative: "bert-base-uncased" (12 layers, 110M) if GPU allows
MAX_LENGTH    = 64          # Headlines are short; 64 tokens is plenty
NUM_LABELS    = 4           # CNN headlines have 4 categories
LR            = 2e-5        # Canonical learning rate for transformer fine-tuning
NUM_EPOCHS    = 2           # Transformers overfit fast on small labeled sets
BATCH_SIZE    = 32          # Fits on T4 with fp16; halve if OOM
WARMUP_STEPS  = 50          # Linear warmup for the first 50 optimizer steps
WEIGHT_DECAY  = 0.01        # AdamW decoupled weight decay
FP16          = torch.cuda.is_available()  # Half-precision on GPU only

print(f"Will fine-tune '{MODEL_NAME}' for {NUM_EPOCHS} epochs with LR={LR}.")

## Section 1 — Why Transformers? The MLP Ceiling

In Notebook 8 you built an MLP that averaged Word2Vec embeddings for each headline, then passed that through a few dense layers. It works. But consider three headlines:

1. *"Stocks soar on strong earnings"*
2. *"Stocks soared after earnings beat"*
3. *"Stocks are soaring following earnings"*

All three headlines convey the same idea. But to your Word2Vec-backed MLP, they look like three **completely different** inputs because:

- "soar", "soared", and "soaring" are three distinct tokens in the vocab → three distinct embedding rows → three distinct mean vectors.
- The MLP has no way to know these are inflections of the same root verb.

A **transformer** solves this through:

1. **Subword tokenization** — "soaring" becomes `['soar', '##ing']` which shares the base form "soar" across tenses.
2. **Contextual embeddings** — the hidden state for "stocks" in sentence 1 is **different** from the hidden state for "stocks" in sentence 2, even though the token is the same, because the surrounding context changed. This is done via **attention**.

That's why BERT-family models beat static-embedding MLPs on tasks where word order, inflection, and context matter.

### Attention intuition (plain English, no math)

Think of a sentence as a room full of people (tokens). In a traditional neural net, each person stands still and never talks to anyone. Their "hidden state" is fixed from the start.

In a transformer, every person (token) looks around the room and asks:

> *"Who should I pay attention to right now?"*

For example, the token "it" in the sentence *"The company announced earnings. **It** beat expectations."* should attend strongly to "company" (2 sentences back!) to know what "it" refers to. Attention computes those weights dynamically: the model learns *"when I see the word 'it', look back for the nearest noun."*

Mathematically, each token has three vectors — Query (Q), Key (K), Value (V). The attention weight between token i and token j is `softmax(Q_i · K_j / sqrt(d))`. You **don't** need to memorize this formula to use BERT, but you **do** need to understand the outcome: every token's hidden state is a weighted mix of **all other tokens** in the sequence. That's why BERT "understands" context.

For classification, we use a special token `[CLS]` (classification) at position 0. By the time the forward pass finishes, `[CLS]`'s hidden state has attended to the entire sentence. We feed that into a linear classifier head.

### BERT at a glance

**BERT** (Bidirectional Encoder Representations from Transformers) is:

- **Encoder-only** — unlike GPT which is decoder-only (left-to-right autoregressive). BERT sees the whole sentence at once (bidirectional).
- **Pretrained on two self-supervised tasks:**
  1. **MLM (Masked Language Model)** — mask out 15% of tokens, ask the model to predict them. This teaches contextual word representations.
  2. **NSP (Next Sentence Prediction)** — given sentences A and B, is B the actual next sentence after A, or a random one? This teaches sentence-level relationships (though modern variants often drop NSP).
- **12 transformer layers** (BERT-base) or **24** (BERT-large). DistilBERT has **6** layers and is 40% smaller / 60% faster with ~97% of BERT-base's quality — that's why we use it in this notebook.

The output is a `(batch, seq_len, hidden_size)` tensor. For classification, we take `hidden_states[:, 0, :]` (the `[CLS]` token) and pass it through a linear layer.

The model has ~66M parameters (DistilBERT) or ~110M (BERT-base). Compare to your MLP from NB8, which had ~500K params. The capacity difference is why BERT wins — but also why it's slower at inference.

### The plan for this notebook

1. **Tokenize** — convert text to WordPiece subword IDs with `AutoTokenizer`. Understand `input_ids`, `attention_mask`, special tokens.
2. **Load pretrained model** — `AutoModelForSequenceClassification.from_pretrained(...)` with `num_labels=4`.
3. **Fine-tune with Trainer** — HuggingFace's high-level API. Configure with `TrainingArguments`, call `trainer.train()`.
4. **Fine-tune manually** — drop below `Trainer` for a PyTorch training loop (for custom losses, curriculum learning, etc.).
5. **Evaluate** — compare with NB8 MLP on the same test set.
6. **Infer** — predict on new headlines; inspect errors; understand when BERT is worth it.

Let's go.

## Section 2 — Tokenization with WordPiece

BERT does **not** tokenize at the word level. It uses **WordPiece**: split rare words into subword units. Example:

- "playing" → `['play', '##ing']`
- "unbelievable" → `['un', '##belie', '##vable']`
- "cat" → `['cat']` (common word, kept whole)

Why subword?

1. **Handles OOV.** If you've never seen "ChatGPT" in training, you can still represent it as `['Chat', '##G', '##PT']`.
2. **Shares roots across inflections.** "run", "running", "runs" all share the subword "run".
3. **Smaller vocab.** ~30K subword tokens vs. ~100K+ words.

HuggingFace's `AutoTokenizer` loads the correct tokenizer for each model. For DistilBERT it's WordPiece; for GPT-2 it's BPE; for LLaMA it's SentencePiece. You don't have to pick — just call `AutoTokenizer.from_pretrained(model_name)` and it does the right thing.

In [ ]:
# Load the tokenizer. First call downloads ~470KB of vocab files from HuggingFace.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer class: {tokenizer.__class__.__name__}")
print(f"Vocab size     : {len(tokenizer)}")

# Tokenize a demo sentence.
demo_text = "Stocks soar on strong earnings report"
demo_tokens = tokenizer(demo_text)

print(f"\nOriginal text: {demo_text}")
print(f"Token IDs    : {demo_tokens['input_ids']}")
print(f"Tokens       : {tokenizer.convert_ids_to_tokens(demo_tokens['input_ids'])}")

### What each field means

The tokenizer returns a dict with keys:

- **`input_ids`**: The sequence of token IDs. Always starts with `[CLS]` (ID 101 for DistilBERT) and ends with `[SEP]` (ID 102).
- **`attention_mask`**: Binary mask — 1 for real tokens, 0 for padding. (Our demo above has no padding, so all 1s.)
- **`token_type_ids`**: Used for sentence-pair tasks (e.g. QA, NLI). DistilBERT doesn't need it for classification; BERT-base does. HuggingFace handles this automatically.

The model will **ignore** any position where `attention_mask == 0`. That's how we handle variable-length inputs in a batch: pad to the same length, then mask out the padding.

In [ ]:
# Subword tokenization in action: a rare word gets split.
rare_word = "unbelievable"
tokens_rare = tokenizer.tokenize(rare_word)
print(f"Word: '{rare_word}' → tokens: {tokens_rare}")

# The '##' prefix means "this is a continuation of the previous subword."
# If you see ['un', '##belie', '##vable'], the model knows to glue them back
# mentally into one semantic unit.

# Another example: an inflection.
inflection = "soaring"
tokens_inflect = tokenizer.tokenize(inflection)
print(f"Word: '{inflection}' → tokens: {tokens_inflect}")

In [ ]:
# Special tokens: [CLS], [SEP], [PAD], [UNK], [MASK].
# - [CLS] = classification token, always at position 0.
# - [SEP] = separator token, marks the end of a sentence.
# - [PAD] = padding token, used to batch variable-length sequences.
# - [UNK] = unknown token (rare with WordPiece, but still exists).
# - [MASK] = used during MLM pretraining; not used in classification.

print("Special token IDs:")
print(f"  [CLS]  = {tokenizer.cls_token} → {tokenizer.cls_token_id}")
print(f"  [SEP]  = {tokenizer.sep_token} → {tokenizer.sep_token_id}")
print(f"  [PAD]  = {tokenizer.pad_token} → {tokenizer.pad_token_id}")
print(f"  [UNK]  = {tokenizer.unk_token} → {tokenizer.unk_token_id}")
print(f"  [MASK] = {tokenizer.mask_token} → {tokenizer.mask_token_id}")

In [ ]:
# Tokenize a batch with padding and truncation.
# padding=True: pad all sequences to the longest in the batch.
# truncation=True: cut sequences longer than max_length.
# max_length=32: cap at 32 tokens (enough for a short demo; we'll use 64 for training).
# return_tensors='pt': return PyTorch tensors instead of lists.

batch_texts = [
    "Short headline",
    "A slightly longer headline about finance",
    "This is an even longer headline with many words about tech and markets",
]

batch_encoded = tokenizer(
    batch_texts,
    padding=True,
    truncation=True,
    max_length=32,
    return_tensors="pt",
)

print(f"Batch input_ids shape    : {batch_encoded['input_ids'].shape}")
print(f"Batch attention_mask shape: {batch_encoded['attention_mask'].shape}")
print(f"\nFirst sequence (with padding):")
print(f"  IDs  : {batch_encoded['input_ids'][0].tolist()}")
print(f"  Mask : {batch_encoded['attention_mask'][0].tolist()}")
print(f"  Tokens: {tokenizer.convert_ids_to_tokens(batch_encoded['input_ids'][0])}")

### 🧪 Lab 1 — Tokenization analysis

Tokenize 5 headlines of varying length with `max_length=32`. Then report:

1. Which headline was **longest** (in tokens) without getting truncated?
2. What's the **padding ratio** for the shortest headline? (number of `[PAD]` tokens / total length)

**Steps:**

1. Pick 5 headlines from the dataset (or make up 5 of varying length: 3 words, 8 words, 15 words, etc.).
2. Tokenize them with `padding=True, truncation=True, max_length=32, return_tensors='pt'`.
3. Inspect `input_ids` to count `[CLS]`, `[SEP]`, real content tokens, and `[PAD]` tokens (ID = 0).
4. Print your findings.

In [ ]:
# 🧪 LAB 1 — Tokenization analysis.

# 1. Define 5 headlines of varying length.
lab_headlines = None  # YOUR CODE (hint: list of 5 strings)

# 2. Tokenize with max_length=32.
lab_encoded = None  # YOUR CODE (hint: tokenizer(..., padding=True, truncation=True, max_length=32, return_tensors='pt'))

# 3. Inspect. Count [CLS] (always 1), [SEP] (always 1), [PAD] (ID 0), and content tokens.
# For the longest: which one has the most content tokens without truncation?
# For the shortest: compute padding_ratio = (number of pad tokens) / 32.

# YOUR CODE: print longest, shortest, padding ratio.
# Example:
# print(f"Longest headline (tokens): ...")
# print(f"Shortest headline padding ratio: ...")

### max_length trade-off

Longer sequences = more memory and compute. For CNN headlines (short), 32–64 is plenty. For long articles (like IMDB reviews in Notebook 10), you may need 128–256.

Rule of thumb:

- **Headlines / tweets:** 32–64
- **Paragraphs:** 128–256
- **Long documents:** 512 (BERT's hard limit) or 1024+ for models like Longformer

DistilBERT maxes out at 512 tokens, though in practice truncating at 128–256 is common for speed.

## Section 3 — Load and Prepare the CNN Headlines Dataset

We'll use the **same** dataset as Notebook 8: CNN headlines with 4 categories (Business, Tech, Entertainment, Health). This ensures an apples-to-apples comparison.

HuggingFace `datasets` library makes this easy:

1. Load from CSV into a pandas DataFrame.
2. Convert to HF `Dataset`.
3. Split into train/val/test (same 70/15/15 split as NB8, stratified).
4. Tokenize the entire dataset in one line with `.map()`.

In [ ]:
# Download the CNN headlines dataset (same as NB8).
CNN_URL = "https://www.dropbox.com/s/0v8d84n2h7xpybx/cnn_headlines.csv?dl=1"
CNN_PATH = "./cnn_headlines.csv"

if not os.path.exists(CNN_PATH):
    print("Downloading CNN headlines dataset (~12MB) ...")
    urllib.request.urlretrieve(CNN_URL, CNN_PATH)
    print("Done.")

# Load into pandas.
df = pd.read_csv(CNN_PATH)
df = df[["headline", "category"]].dropna().reset_index(drop=True)

# Map category names to integer labels.
categories = sorted(df["category"].unique())
label2id = {cat: i for i, cat in enumerate(categories)}
id2label = {i: cat for cat, i in label2id.items()}
df["label"] = df["category"].map(label2id)

print(f"Dataset size: {len(df):,} headlines")
print(f"Categories  : {categories}")
print(f"\nSample rows:")
print(df.head())

In [ ]:
# Convert to HuggingFace Dataset.
dataset = Dataset.from_pandas(df[["headline", "label"]])
print(f"HF Dataset: {dataset}")
print(f"\nFirst example:")
print(dataset[0])

In [ ]:
# Split into train / val / test (70% / 15% / 15%), stratified by label.
# This matches the split in NB8 for fair comparison.
train_val_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df["label"], random_state=SEED
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.15 / 0.85, stratify=train_val_df["label"], random_state=SEED
)

# Convert to HF Dataset splits.
train_dataset = Dataset.from_pandas(train_df[["headline", "label"]].reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df[["headline", "label"]].reset_index(drop=True))
test_dataset  = Dataset.from_pandas(test_df[["headline", "label"]].reset_index(drop=True))

print(f"Train: {len(train_dataset):,} | Val: {len(val_dataset):,} | Test: {len(test_dataset):,}")

In [ ]:
# Define a tokenize function for .map().
# batched=True means the function receives a batch of examples (dict of lists).
def tokenize_function(examples):
    return tokenizer(
        examples["headline"],
        padding=False,        # Don't pad yet; we'll pad dynamically per batch later
        truncation=True,
        max_length=MAX_LENGTH,
    )

# Tokenize all three splits. This takes ~10–20 seconds.
# The tokenized columns (input_ids, attention_mask) are added to the dataset.
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset   = val_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function, batched=True)

print(f"Tokenized train dataset: {train_dataset}")
print(f"Columns: {train_dataset.column_names}")

### Why `padding=False` in the tokenize function?

We **don't** pad during `.map()` because different batches may have different max lengths. Instead, we'll use `DataCollatorWithPadding` to pad **dynamically** per batch at training time. This saves memory and compute — no point padding a batch of 3-word headlines to 64 tokens if the longest in that batch is 10 tokens.

## Section 4 — Load the Pretrained Model

`AutoModelForSequenceClassification` loads a model with a classification head on top. For DistilBERT, the architecture is:

```
Input → DistilBERT (6 transformer layers) → [CLS] hidden state → Dropout → Linear(768 → num_labels)
```

The **backbone** (6 layers) is pretrained on MLM. The **head** (Linear layer) is **randomly initialized** — we're training it from scratch on our CNN headlines task. The backbone gets fine-tuned too, but with a much smaller learning rate (2e-5) so we don't destroy the pretrained knowledge.

In [ ]:
# Load DistilBERT for sequence classification with 4 output classes.
# First call downloads ~270MB of model weights.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

# Move to device (GPU if available).
model.to(device)

print(f"Model class: {model.__class__.__name__}")
print(f"Num params : {sum(p.numel() for p in model.parameters()):,}")
print(f"Num layers : {model.config.num_hidden_layers}")
print(f"Hidden size: {model.config.hidden_size}")

In [ ]:
# One forward pass pre-training to see what random predictions look like.
# The model hasn't learned the task yet, so output should be ~uniform over 4 classes.
sample_batch = tokenizer(
    ["Tech stocks rally", "New health guidelines released"],
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
).to(device)

model.eval()
with torch.no_grad():
    outputs = model(**sample_batch)

logits = outputs.logits
probs = torch.softmax(logits, dim=-1)
preds = torch.argmax(logits, dim=-1)

print("Pre-training predictions (should be near-random):")
for i, text in enumerate(["Tech stocks rally", "New health guidelines released"]):
    print(f"  '{text}' → pred={id2label[preds[i].item()]} (probs={probs[i].cpu().numpy()})")

### Fine-tuning vs. training from scratch

- **Training from scratch:** Initialize all weights randomly, train on your task. Requires millions of examples to get good. Only viable for Google / Meta.
- **Fine-tuning:** Start with pretrained weights (learned from billions of tokens of text), then **update** them on your task. Works with 1K–10K examples.

What gets updated?

- **The classification head** (Linear layer) — always updated, starts random.
- **The backbone** (transformer layers) — updated, but with a small LR (2e-5) to avoid catastrophic forgetting of pretrained knowledge.

You can also **freeze** the backbone and only train the head (faster, lower memory, but usually lower quality). We'll train end-to-end in this notebook.

## Section 5 — Fine-tuning with the Trainer API

HuggingFace `Trainer` is a high-level wrapper that handles:

- Training loop (forward, backward, optimizer step, logging)
- Evaluation loop
- Checkpointing (save best model based on val metric)
- Multi-GPU / fp16 via `accelerate`

You just configure `TrainingArguments`, pass your datasets, and call `trainer.train()`. Under the hood it's doing the same thing as a manual PyTorch loop — we'll show that in Section 6.

In [ ]:
# Define a compute_metrics function for evaluation.
# The Trainer will call this on the val set after each epoch.
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1": f1}

print("Metrics function ready.")

In [ ]:
# Configure TrainingArguments.
# - output_dir: where to save checkpoints.
# - num_train_epochs: 2 is typical for transformer fine-tuning on small labeled sets.
# - per_device_train_batch_size: 32 fits on T4 GPU with fp16; halve if OOM.
# - learning_rate: 2e-5 is canonical for transformer fine-tuning (100× smaller than MLP!).
# - weight_decay: 0.01 for AdamW.
# - eval_strategy='epoch': evaluate after each epoch.
# - save_strategy='epoch': save checkpoint after each epoch.
# - load_best_model_at_end=True: after training, reload the best checkpoint (by f1).
# - metric_for_best_model='f1': pick the checkpoint with highest macro-F1.
# - fp16=True: use half-precision on GPU for 2× speedup (only on GPU).
# - logging_steps: log every N steps (default 500; we'll use 100 for visibility).

training_args = TrainingArguments(
    output_dir="./bert_cnn_headlines",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    fp16=FP16,
    seed=SEED,
)

print(training_args)

In [ ]:
# Create a DataCollatorWithPadding to pad batches dynamically.
# This avoids padding all sequences to MAX_LENGTH upfront.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Instantiate the Trainer.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer ready. Call trainer.train() to start fine-tuning.")

In [ ]:
# Fine-tune the model. This takes ~15 min on Colab T4, ~3 hours on CPU.
# Watch the progress bar: you should see loss decreasing and val F1 increasing.
trainer.train()

print("\nFine-tuning complete! The Trainer has automatically loaded the best checkpoint.")

In [ ]:
# Evaluate on the held-out test set.
test_results = trainer.evaluate(test_dataset)
print(f"\n=== Test Set Results ===")
print(f"Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"Macro-F1: {test_results['eval_f1']:.4f}")

### Compare with Notebook 8 MLP

In Notebook 8, your MLP achieved ~92% accuracy on this same test set. BERT should hit ~95–96%. The gain comes from:

1. **Contextual embeddings** — "soar" and "soared" share representations.
2. **Attention** — the model learns which words matter most for classification.
3. **Pretraining** — 66M parameters trained on billions of tokens give you a strong starting point.

Trade-off: BERT is **slower** (10–20ms per headline on GPU vs. <1ms for MLP) and **heavier** (270MB model vs. 2MB). For a production system, you'd weigh accuracy vs. latency vs. cost.

### 🧪 Lab 2 — Experiment with num_train_epochs

The default is 2 epochs. Try 3 epochs and re-train. Does it improve test F1, or does the model overfit?

**Steps:**

1. Change `NUM_EPOCHS` to 3.
2. Re-instantiate the model (fresh weights) and `TrainingArguments`.
3. Create a new `Trainer` and call `trainer.train()`.
4. Evaluate on the test set.
5. Compare: did 3 epochs beat 2 epochs, or did val F1 start dropping (overfitting)?

> Hint: You can also check `trainer.state.log_history` to see the val F1 per epoch. If epoch 3's val F1 is **lower** than epoch 2's, you overfit.

In [ ]:
# 🧪 LAB 2 — Experiment with num_train_epochs.

# 1. Change NUM_EPOCHS to 3 (or keep it 2 if you want to compare side-by-side).
NUM_EPOCHS_LAB = None  # YOUR CODE (e.g., 3)

# 2. Re-load the model (fresh weights) and configure TrainingArguments.
# model_lab = AutoModelForSequenceClassification.from_pretrained(...).to(device)
# training_args_lab = TrainingArguments(...)

# 3. Create a new Trainer.
# trainer_lab = Trainer(...)

# 4. Train.
# trainer_lab.train()

# 5. Evaluate on test.
# test_results_lab = trainer_lab.evaluate(test_dataset)
# print(f"Test accuracy (3 epochs): {test_results_lab['eval_accuracy']:.4f}")
# print(f"Test macro-F1 (3 epochs): {test_results_lab['eval_f1']:.4f}")

# YOUR CODE

## Section 6 — Manual PyTorch Fine-tuning Loop

The `Trainer` API is great for 90% of use cases. But sometimes you need:

- Custom loss functions (e.g. focal loss, label smoothing)
- Curriculum learning (train on easy examples first)
- Custom logging / callbacks
- Manual control over optimizer / scheduler

For these, you drop below `Trainer` and write a PyTorch training loop. The pattern is **identical** to Notebook 8, just swap in the BERT model and use AdamW + a linear warmup scheduler.

In [ ]:
# Load a fresh model (so we're not contaminating with the Trainer's weights).
model_manual = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
).to(device)

print(f"Fresh model loaded with {sum(p.numel() for p in model_manual.parameters()):,} params.")

In [ ]:
# Wrap the HF Dataset in a PyTorch DataLoader with dynamic padding.
# DataCollatorWithPadding pads each batch to the max length in that batch.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# AdamW optimizer (not Adam — AdamW has decoupled weight decay, canonical for transformers).
optimizer = torch.optim.AdamW(model_manual.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Linear warmup scheduler: LR ramps up linearly for the first WARMUP_STEPS, then decays linearly to 0.
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps,
)

print(f"Optimizer: AdamW with LR={LR}, weight_decay={WEIGHT_DECAY}")
print(f"Scheduler: Linear warmup for {WARMUP_STEPS} steps, then linear decay over {total_steps} total steps.")

In [ ]:
# Manual training loop — same pattern as NB8, just swap model and optimizer.
loss_fn = nn.CrossEntropyLoss()

for epoch in range(NUM_EPOCHS):
    model_manual.train()
    train_loss = 0.0
    
    for batch in train_loader:
        # Move batch to device.
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Forward pass.
        outputs = model_manual(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss  # The model computes loss internally if labels are provided
        
        # Backward pass.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    
    # Validation loop.
    model_manual.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]
            
            outputs = model_manual(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            
            val_preds.extend(preds)
            val_labels.extend(labels.numpy())
    
    val_acc = accuracy_metric.compute(predictions=val_preds, references=val_labels)["accuracy"]
    val_f1 = f1_metric.compute(predictions=val_preds, references=val_labels, average="macro")["f1"]
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} — Train loss: {avg_train_loss:.4f} | Val acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

print("\nManual training complete!")

### 🧪 Lab 3 — Manual loop with early stopping

The loop above trains for a fixed number of epochs. In production, you'd add **early stopping**: if val F1 doesn't improve for N epochs, stop training to avoid overfitting.

**Task:** Modify the loop above to track the best val F1 and stop if it doesn't improve for 2 consecutive epochs (patience=2).

**Steps:**

1. Before the loop, initialize `best_val_f1 = 0` and `patience_counter = 0`.
2. After each epoch, check if `val_f1 > best_val_f1`. If yes, update `best_val_f1` and reset `patience_counter = 0`. If no, increment `patience_counter`.
3. If `patience_counter >= 2`, break the loop early.

Fill in the skeleton below.

In [ ]:
# 🧪 LAB 3 — Manual loop with early stopping.

# Load a fresh model.
# model_early = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS, ...).to(device)
# optimizer_early = torch.optim.AdamW(...)
# scheduler_early = get_linear_schedule_with_warmup(...)

# Early stopping state.
# best_val_f1 = 0
# patience_counter = 0
# patience = 2

# for epoch in range(10):  # max 10 epochs, but we'll stop early if no improvement
#     # ... training loop ...
#     # ... validation loop ...
#     # if val_f1 > best_val_f1:
#     #     best_val_f1 = val_f1
#     #     patience_counter = 0
#     # else:
#     #     patience_counter += 1
#     # if patience_counter >= patience:
#     #     print(f"Early stopping at epoch {epoch+1}")
#     #     break

# YOUR CODE

## Section 7 — Inference and Error Analysis

Now that we have a fine-tuned model, let's:

1. Use HuggingFace `pipeline` for easy inference.
2. Test on new headlines (including adversarial examples from NB8).
3. Plot a confusion matrix to see which classes BERT still confuses.
4. Sample and inspect errors — are they genuinely ambiguous, or systematic bugs?

In [ ]:
# Build an inference pipeline with the fine-tuned model.
from transformers import pipeline

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# Test on 5 new headlines.
new_headlines = [
    "Apple announces new iPhone with AI features",
    "Fed raises interest rates by 0.25%",
    "New cancer treatment shows promising results in trials",
    "Box office records shattered by latest blockbuster",
    "Startup raises $100M Series B led by Sequoia",  # Ambiguous: could be Tech or Business
]

for headline in new_headlines:
    result = classifier(headline, top_k=1)[0]
    print(f"{headline:<60} → {result['label']} ({result['score']:.3f})")

In [ ]:
# Confusion matrix on the test set.
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=categories, yticklabels=categories, square=True)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("BERT Confusion Matrix — CNN Headlines Test Set")
plt.tight_layout()
plt.show()

print("\nClassification report:")
print(classification_report(all_labels, all_preds, target_names=categories))

### What classes does BERT still confuse?

Look at the off-diagonal cells in the confusion matrix. Typical confusions:

- **Tech ↔ Business** — headlines like *"Apple stock soars"* or *"Amazon reports quarterly earnings"* are genuinely ambiguous. Is it a tech story or a business story? Both. The dataset labels are noisy here.
- **Health ↔ Entertainment** — rare, but headlines like *"Celebrity opens wellness clinic"* could go either way.

If you see systematic errors (e.g. all Health headlines predicted as Entertainment), that's a bug — check your label mapping, tokenization, or class balance.

In [ ]:
# Error analysis: sample 5 misclassifications per class.
errors = []
for i, (pred, true) in enumerate(zip(all_preds, all_labels)):
    if pred != true:
        headline = test_dataset[i]["headline"]
        errors.append((headline, id2label[true], id2label[pred]))

print(f"Total errors: {len(errors)} / {len(all_labels)} ({100 * len(errors) / len(all_labels):.1f}%)")
print(f"\nSample errors:")
for headline, true_label, pred_label in errors[:10]:
    print(f"  True={true_label:<15} Pred={pred_label:<15} | {headline}")

### When is BERT worth it?

| Scenario | MLP (NB8) | BERT (NB9) |
|----------|-----------|------------|
| Accuracy | ~92% | ~95% |
| Training time | 5–10 min | 15–30 min |
| Inference latency (GPU) | <1ms | 10–20ms |
| Model size | 2 MB | 270 MB |
| Labeled data needed | 5K+ | 1K+ (pretrained = data-efficient) |
| When to use | Simple intent, latency-critical, small budget | High accuracy needed, context matters, have GPU |

**Use BERT when:**

- You have >1K labeled examples and want the best accuracy.
- Word order and phrasing matter (e.g. *"not good"* vs *"good"*).
- You can afford the inference cost (GPU, or batch heavily on CPU).

**Use MLP when:**

- You need <5ms latency.
- You're deploying to mobile / edge devices (limited memory).
- Accuracy in the low 90s is good enough.

## Section 8 — Save, Reload, and Deployment

HuggingFace makes it trivial to save and reload models. You can also push to the HuggingFace Hub for public sharing (optional).

In [ ]:
# Save the model and tokenizer to disk.
model.save_pretrained("./bert_cnn_headlines_final")
tokenizer.save_pretrained("./bert_cnn_headlines_final")
print("Model and tokenizer saved to ./bert_cnn_headlines_final/")

In [ ]:
# Reload and verify it works.
model_reloaded = AutoModelForSequenceClassification.from_pretrained("./bert_cnn_headlines_final").to(device)
tokenizer_reloaded = AutoTokenizer.from_pretrained("./bert_cnn_headlines_final")

# Quick test.
test_headline = "Tech giant launches new AI assistant"
inputs = tokenizer_reloaded(test_headline, return_tensors="pt").to(device)
model_reloaded.eval()
with torch.no_grad():
    outputs = model_reloaded(**inputs)
pred_id = torch.argmax(outputs.logits, dim=-1).item()
print(f"Test: '{test_headline}' → {id2label[pred_id]}")

### (Optional) Push to HuggingFace Hub

If you have a HuggingFace account and want to share your model:

```python
# 1. Log in (you'll be prompted for your token).
# from huggingface_hub import notebook_login
# notebook_login()

# 2. Push to hub.
# model.push_to_hub("your-username/bert-cnn-headlines")
# tokenizer.push_to_hub("your-username/bert-cnn-headlines")
```

Then anyone can load your model with:

```python
from transformers import AutoModelForSequenceClassification, AutoTokenizer
model = AutoModelForSequenceClassification.from_pretrained("your-username/bert-cnn-headlines")
tokenizer = AutoTokenizer.from_pretrained("your-username/bert-cnn-headlines")
```

### Deployment tips

1. **Quantization** — reduce model size and latency with INT8 quantization (via `optimum` or ONNX Runtime). Can get 2–4× speedup with minimal accuracy loss.
2. **Batch inference** — don't process one headline at a time. Batch 32–64 for much higher throughput.
3. **Distillation** — if DistilBERT is still too slow, distill it further into a 3-layer model (DistilDistilBERT?). Or switch to MobileBERT / TinyBERT.
4. **Cache embeddings** — if your headlines don't change often, precompute and cache the `[CLS]` embeddings, then just run a linear classifier at inference time.

## Wrap-up

### What you can now ship

- A transformer-based text classifier that beats static-embedding MLPs by 3–4 points on most NLP tasks.
- The ability to fine-tune **any** HuggingFace model (BERT, RoBERTa, DeBERTa, ELECTRA, etc.) by swapping `MODEL_NAME`.
- A pattern you can apply to any classification problem: sentiment, intent, topic, toxicity, etc.

### Key takeaways

| Concept | What to remember |
|---------|------------------|
| **Attention** | Tokens "talk to each other" and re-weight each other's features dynamically. |
| **Subword tokenization** | WordPiece splits rare words into subwords, shares roots across inflections. |
| **Fine-tuning LR** | 2e-5 is canonical for transformers — 100× smaller than MLP LR. |
| **Trainer API** | High-level wrapper for 90% of use cases; drop below for custom logic. |
| **When to use BERT** | >1K labeled examples, context matters, have GPU budget. |
| **When to use MLP** | <5ms latency, edge deployment, accuracy in low 90s is OK. |

### Common pitfalls (memorize these)

1. **`num_labels=4` must match your class count.** Silent bug if wrong.
2. **Learning rate.** Transformer fine-tune LR ≈ 2e-5 is 100× smaller than MLP LR. If you leave it at 1e-3, you'll get NaN loss or catastrophic forgetting.
3. **Batch size vs. GPU memory.** DistilBERT at `max_length=64` needs ~2GB per batch of 32 with fp16. If OOM, halve batch size and use `gradient_accumulation_steps=2`.
4. **fp16 on CPU doesn't work.** Guard with `fp16=torch.cuda.is_available()`.
5. **`load_best_model_at_end=True` only works if `eval_strategy` and `save_strategy` match and `save_total_limit >= 2`.**
6. **Tokenizer cache on Colab.** First run downloads ~270MB. Warn students.

### Self-check quiz

1. You have 500 labeled examples. Would you fine-tune BERT or use LogReg? Why?
2. Why is the fine-tuning LR 100× smaller than MLP's?
3. Your BERT test acc is 60%, LogReg test acc is 90%. Name 3 likely bugs.
4. Trainer's `load_best_model_at_end` silently loaded the *first-epoch* checkpoint. What did you forget to configure?

**Answers:**

1. **LogReg.** 500 examples is borderline — BERT may overfit. Try LogReg first; if it plateaus around 80–85%, *then* try BERT with heavy regularization (dropout=0.5, weight_decay=0.1).
2. Pretrained weights are already good — a large LR would destroy them (catastrophic forgetting). You're nudging, not retraining from scratch.
3. (a) Wrong `num_labels`, (b) Labels not aligned with model output, (c) LR too high → NaN or divergence, (d) Forgot to call `model.train()` / `model.eval()`.
4. You forgot to set both `eval_strategy='epoch'` AND `save_strategy='epoch'`. If they don't match, the Trainer can't find the best checkpoint.

### What's next

Notebook 10 (Capstone 2) is the IMDB sentiment challenge. You'll pick **either** the MLP path (NB8 style) **or** the BERT path (NB9 style), build end-to-end, and write an engineering memo justifying your choice with hard numbers on accuracy, latency, and memory.

— *Notebook 9 complete.* 🚀